In [0]:
%sh
pip install gspread pymongo[srv]

In [0]:
%sh pwd

In [0]:
import sys
sys.path.append("/Workspace/Users/matumazparrote@gmail.com/elt_products_scraping")

In [0]:
import json
from google.oauth2 import service_account
import os
import pandas as pd
import gspread
from google.oauth2.service_account import Credentials
from src.config.mongo_db import mongo_db
from src.utils.google_sheet import google_sheets_handler
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

creds_json = dbutils.secrets.get(scope="gcp-creds", key="google-credentials")
creds_dict = json.loads(creds_json)
# print(f"CREDENCIALES DE GOOGLE: {creds_dict}")
print("Credentials loaded successfully")
GOOGLE_SCOPES = [
    "https://www.googleapis.com/auth/spreadsheets",
    "https://www.googleapis.com/auth/drive"
]

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS products")

In [0]:
product_catalog = google_sheets_handler.read(creds_dict, GOOGLE_SCOPES)
product_catalog_spark = spark.createDataFrame(product_catalog)
product_catalog_spark.write.mode("overwrite").saveAsTable("products.catalog")

In [0]:
spark.sql("SHOW TABLES IN products").show()

In [0]:
mongo_db.connect(dbutils.secrets.get("mondo_db_creds", "MONGO_DB_URI"), db_name="products")

In [0]:
scraped_products_collection = mongo_db.get_database()["scraped_products"]
scraped_products = scraped_products_collection.find()
df_scraped_products = pd.DataFrame(scraped_products).drop(["_id"], axis=1)
df_scraped_products["scraped_at"] = df_scraped_products["scraped_at"].apply(
    lambda x: f"{x} 00:00:00" if len(str(x)) == 10 else str(x)
)
df_scraped_products["scraped_at"] = pd.to_datetime(df_scraped_products["scraped_at"], format="%Y-%m-%d %H:%M:%S")
df_scraped_products["raw_data"] = df_scraped_products["raw_data"].apply(lambda x: json.dumps(x) if isinstance(x, dict) else x)

# spark_raw_scrapping_data = spark.createDataFrame(df_scraped_products)
# spark_raw_scrapping_data.write.mode("overwrite").saveAsTable("products.raw_scrapping_data")
# df_scraped_products.head()



new_data_df = spark.createDataFrame(df_scraped_products)

# 2. Verificar si la tabla existe para hacer MERGE o crearla por primera vez
if spark.catalog.tableExists("products.raw_scrapping_data"):
    print("LA TABLA EXISTE")
    deltaTable = DeltaTable.forName(spark, "products.raw_scrapping_data")
    
    (deltaTable.alias("target")
      .merge(
        new_data_df.alias("source"),
        # Claves de negocio para identificar duplicados
        "target.product_id = source.product_id AND target.retailer = source.retailer AND target.scraped_at = source.scraped_at"
      )
      .whenMatchedUpdateAll() # Si ya existe para ese día/tienda, actualiza (por si cambió el JSON)
      .whenNotMatchedInsertAll() # Si es nuevo, inserta
      .execute())
    print("SE REALIZA EL MERGE CON EL UPSERT")
else:
    # Si la tabla no existe, la crea
    new_data_df.write.format("delta").saveAsTable("products.raw_scrapping_data")
    print("SE CREA LA TABLA")

In [0]:
# # 1. Cargar errores desde Mongo
# error_logs_collection = mongo_db.get_database()["error_logs"]
# error_logs = error_logs_collection.find()
# df_error_logs = pd.DataFrame(error_logs).drop(["_id"], axis=1)

# # 2. Normalizar timestamp
# df_error_logs["timestamp"] = df_error_logs["timestamp"].apply(
#     lambda x: f"{x} 00:00:00" if len(str(x)) == 10 else str(x)
# )
# df_error_logs["timestamp"] = pd.to_datetime(df_error_logs["timestamp"], format="%Y-%m-%d %H:%M:%S")

# # 3. Convertir a Spark DataFrame
# spark_errors_df = spark.createDataFrame(df_error_logs)
# spark_errors_df.write.format("delta").saveAsTable("products.logs_errors_data")
# print("Tabla de errores creada con todos los registros iniciales")




# 1. Obtener el último timestamp cargado en Databricks
try:
    last_error_ts = spark.sql("SELECT max(timestamp) FROM products.logs_errors_data").collect()[0][0]
    print(f"Buscando errores posteriores a: {last_error_ts}")
except Exception:
    last_error_ts = None
    print("No se encontró tabla previa. Iniciando carga completa.")

# 2. Consultar Mongo solo por errores nuevos (Filtro en origen para ahorrar memoria)
query = {"timestamp": {"$gt": last_error_ts}} if last_error_ts else {}
error_logs_collection = mongo_db.get_database()["error_logs"]
new_error_logs = list(error_logs_collection.find(query))

if new_error_logs:
    # 3. Procesamiento y Normalización (Pandas)
    df_new_errors = pd.DataFrame(new_error_logs).drop(["_id"], axis=1)
    df_new_errors["timestamp"] = pd.to_datetime(df_new_errors["timestamp"])
    
    # Convertir a Spark
    spark_new_errors_df = spark.createDataFrame(df_new_errors)

    # 4. Upsert (MERGE) para evitar duplicados si se corre dos veces el mismo día
    if spark.catalog.tableExists("products.logs_errors_data"):
        deltaTable = DeltaTable.forName(spark, "products.logs_errors_data")
        
        merge_result = (deltaTable.alias("target")
          .merge(
            spark_new_errors_df.alias("source"),
            "target.product_id = source.product_id AND \
             target.retailer = source.retailer AND \
             target.timestamp = source.timestamp"
          )
          .whenNotMatchedInsertAll()
          .execute())
        
        # 5. Contabilizar registros nuevos
        inserted_rows = spark.sql("SELECT operationMetrics.numTargetRowsInserted FROM (DESCRIBE HISTORY products.logs_errors_data LIMIT 1)").collect()[0][0]
        print(f"Carga incremental finalizada. Se insertaron {inserted_rows} registros nuevos.")
    else:
        # Carga inicial si no existe
        spark_new_errors_df.write.format("delta").saveAsTable("products.logs_errors_data")
        print(f"Tabla creada con {spark_new_errors_df.count()} registros.")
else:
    print("No se encontraron errores nuevos en MongoDB.")
